# Heston Continuous Baseline

Refactored continuous TC-VAE baseline demo for the Heston stochastic-volatility benchmark. This notebook does not require released checkpoints by default and does not run training unless `RUN_SMOKE=True`. Generated artefacts should stay under ignored `outputs/` paths.


## Setup

Load the package, locate the repository root, and display the selected continuous config.


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Markdown, display

RUN_SMOKE = False
RUN_TRAINING = False
RUN_EVALUATION = False


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_yaml(path: str | Path) -> dict[str, Any]:
    resolved = repo_path(path)
    if not resolved.exists():
        return {}
    loaded = yaml.safe_load(resolved.read_text())
    return loaded if isinstance(loaded, dict) else {}


def load_json(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


def print_command(command: list[str]) -> None:
    print(" ".join(shlex.quote(part) for part in command))


def maybe_run(command: list[str], *, enabled: bool, label: str) -> None:
    print(f"{label} command:")
    print_command(command)
    if enabled:
        subprocess.run(command, cwd=REPO_ROOT, check=True)
    else:
        print(f"{label} skipped; enable the matching RUN_* flag to execute it.")


print(f"Repository root: {REPO_ROOT}")
CONFIG_PATH = "configs/experiments/heston_info_cvae.yaml"
OUTPUT_DIR = "outputs/continuous/heston_info_cvae"
BASE_DATA_DIR = "data/processed"
N_SAMPLE_TEST = 1024
print(f"Config: {display_path(CONFIG_PATH)}")

## Configuration Summary


In [ ]:
raw_config = load_yaml(CONFIG_PATH)
rows = []
for section, value in raw_config.items():
    if isinstance(value, dict):
        for key, item in value.items():
            if isinstance(item, (str, int, float, bool)) or item is None:
                rows.append({"section": section, "field": key, "value": item})
    elif isinstance(value, (str, int, float, bool)) or value is None:
        rows.append({"section": "root", "field": section, "value": value})
if rows:
    display(pd.DataFrame(rows))
else:
    print(f"Config missing or empty: {display_path(CONFIG_PATH)}")

## Dry-Run Commands

The smoke command builds data and model wiring without performing full training. The evaluation command is printed for use after a local continuous checkpoint exists.


In [ ]:
train_smoke_command = [
    "poetry",
    "run",
    "tcvae-train",
    "--config",
    display_path(CONFIG_PATH),
    "--output-dir",
    display_path(OUTPUT_DIR),
    "--epochs",
    "1",
    "--no-wandb",
    "--dry-run",
]
evaluation_command = [
    "poetry",
    "run",
    "tcvae-evaluate",
    "--config",
    display_path(CONFIG_PATH),
    "--model-dir",
    display_path(Path(OUTPUT_DIR) / "<training-run>" / "final_model"),
    "--output-dir",
    display_path(Path(OUTPUT_DIR) / "evaluation"),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--n-sample-test",
    str(N_SAMPLE_TEST),
    "--seed",
    "99",
]
maybe_run(train_smoke_command, enabled=RUN_SMOKE, label="Continuous smoke")
print("\nEvaluation command for an existing checkpoint:")
print_command(evaluation_command)

## Expected Outputs

For a completed local run, keep generated summaries, figures, checkpoints, and executed notebooks below `outputs/`. The committed notebook should remain output-stripped.
